## Notebook to look at structure of LOBSTER data

In [18]:
# Load AMZN data
from pathlib import Path

# Data lives OUTSIDE the repo. expanduser() turns "~" into /Users/angusslater
DATA_DIR = Path("~/Desktop/LOBSTER").expanduser()
AMZN_DIR = DATA_DIR / "LOBSTER_SampleFile_AMZN_2012-06-21_10"

msg_path = AMZN_DIR / "AMZN_2012-06-21_34200000_57600000_message_10.csv"
ob_path  = AMZN_DIR / "AMZN_2012-06-21_34200000_57600000_orderbook_10.csv"

print(msg_path.exists(), ob_path.exists())

True True


In [19]:
# Read both files into pandas and join them into one big DataFrame
import pandas as pd

# Message file: one row per event
msg_cols = ["time", "type", "order_id", "size", "price", "direction"]

# Orderbook file: 10 levels x [ask price, ask size, bid price, bid size]
ob_cols = []
for level in range(1, 11):
    ob_cols += [f"ask_p{level}", f"ask_s{level}", f"bid_p{level}", f"bid_s{level}"]

# Neither file has a header row
msg = pd.read_csv(msg_path, header=None, names=msg_cols)
ob = pd.read_csv(ob_path, header=None, names=ob_cols)

# Row k of the orderbook is the book after message k, so join side by side
lob = pd.concat([msg, ob], axis=1)

print(lob.shape)
lob.head()

(269748, 46)


,time,type,order_id,size,price,direction,ask_p1,ask_s1,bid_p1,bid_s1,...,bid_p8,bid_s8,ask_p9,ask_s9,bid_p9,bid_s9,ask_p10,ask_s10,bid_p10,bid_s10
0,34200.017460,5,0,1,2238200,-1,2239500,100,2231800,100,...,2202500,5000,2294300,100,2202000,100,2298000,100,2189700,100
1,34200.189608,1,11885113,21,2238100,1,2239500,100,2238100,21,...,2204000,100,2294300,100,2202500,5000,2298000,100,2202000,100
2,34200.189608,1,3911376,20,2239600,-1,2239500,100,2238100,21,...,2204000,100,2267700,100,2202500,5000,2294300,100,2202000,100
3,34200.189608,1,11534792,100,2237500,1,2239500,100,2238100,21,...,2213000,4000,2267700,100,2204000,100,2294300,100,2202500,5000
4,34200.189608,1,1365373,13,2240000,-1,2239500,100,2238100,21,...,2213000,4000,2267700,100,2204000,100,2294300,100,2202500,5000


In [20]:
# Basic health checks
print("missing values:", lob.isna().sum().sum())
print("time range:", lob["time"].min(), "to", lob["time"].max())
print("time sorted:", lob["time"].is_monotonic_increasing)

# How many of each event type (1 submit, 2 partial cancel, 3 delete, 4 visible exec, 5 hidden exec)
lob["type"].value_counts().sort_index()

missing values: 0
time range: 34200.017459617 to 57599.95935965
time sorted: True


type
1    131954
2      2917
3    123458
4      8974
5      2445
Name: count, dtype: int64

In [21]:
# Do the real data obey the invariants we want TimeGAN to learn?
ask_p = lob[[f"ask_p{i}" for i in range(1, 11)]]
bid_p = lob[[f"bid_p{i}" for i in range(1, 11)]]
sizes = lob[[f"{side}_s{i}" for side in ("ask", "bid") for i in range(1, 11)]]

# Each line counts the rows that BREAK a rule, so 0 is good
print("crossed book (bid1 >= ask1):", (lob["bid_p1"] >= lob["ask_p1"]).sum())
print("ask prices not increasing:  ", (ask_p.diff(axis=1).iloc[:, 1:] <= 0).any(axis=1).sum())
print("bid prices not decreasing:  ", (bid_p.diff(axis=1).iloc[:, 1:] >= 0).any(axis=1).sum())
print("negative sizes:             ", (sizes < 0).any(axis=1).sum())

crossed book (bid1 >= ask1): 0
ask prices not increasing:   0
bid prices not decreasing:   0
negative sizes:              0


In [24]:
# Check for dummy (placeholder) values. LOBSTER fills an empty book level with
# ask price 9999999999, bid price -9999999999 and size 0, so those are what to look for.
price_cols = [f"{side}_p{i}" for side in ("ask", "bid") for i in range(1, 11)]
size_cols = [f"{side}_s{i}" for side in ("ask", "bid") for i in range(1, 11)]
prices = lob[price_cols]
sizes = lob[size_cols]

# Each line counts bad values, so 0 is good
print("book prices that are 0 or negative:   ", (prices <= 0).sum().sum())
print("book prices of 9999999999 (dummy):    ", (prices.abs() >= 9_999_999_999).sum().sum())
print("book prices not on a 1 cent tick:     ", (prices % 100 != 0).sum().sum())
print("book sizes that are 0 or negative:    ", (sizes <= 0).sum().sum())
print("message prices that are 0 or negative:", (lob["price"] <= 0).sum())
print("message sizes that are 0 or negative: ", (lob["size"] <= 0).sum())
print("directions that are not 1 or -1:      ", (~lob["direction"].isin([1, -1])).sum())
print("trading halts (type 7):               ", (lob["type"] == 7).sum())

# The one placeholder LOBSTER does use here: hidden executions (type 5) get order ID 0
print("\norder IDs of 0:", (lob["order_id"] == 0).sum(), "of which type 5:", ((lob["order_id"] == 0) & (lob["type"] == 5)).sum())

# A real price should be near the mid-price. A dummy would be thousands of dollars away.
mid_raw = (lob["ask_p1"] + lob["bid_p1"]) / 2
furthest = prices.sub(mid_raw, axis=0).abs().max().max() / 10000
print(f"furthest any book price is from the mid: ${furthest:.2f}")

# The smallest and largest value in every price and size column, to eyeball
summary = pd.DataFrame({
    "min price $": (prices.min() / 10000).values, "max price $": (prices.max() / 10000).values,
    "min size": sizes.min().values, "max size": sizes.max().values,
}, index=[c.replace("_p", " level ") for c in price_cols])
summary

book prices that are 0 or negative:    0
book prices of 9999999999 (dummy):     0
book prices not on a 1 cent tick:      0
book sizes that are 0 or negative:     0
message prices that are 0 or negative: 0
message sizes that are 0 or negative:  0
directions that are not 1 or -1:       0
trading halts (type 7):                0

order IDs of 0: 2445 of which type 5: 2445
furthest any book price is from the mid: $6.24


,min price $,max price $,min size,max size
ask level 1,220.53,226.04,1,5700
ask level 2,220.59,226.06,1,3250
ask level 3,220.62,226.10,1,2815
ask level 4,220.63,226.14,1,2799
ask level 5,220.64,226.17,1,2799
ask level 6,220.66,226.18,1,2999
ask level 7,220.67,226.19,1,2999
ask level 8,220.68,226.77,1,2999
ask level 9,220.69,229.43,1,3100
ask level 10,220.72,229.80,1,3104


In [22]:
# Spread and mid-price in dollars (prices are stored as USD x 10,000)
lob["spread"] = (lob["ask_p1"] - lob["bid_p1"]) / 10000
lob["mid"] = (lob["ask_p1"] + lob["bid_p1"]) / 2 / 10000

lob[["spread", "mid"]].describe()

,spread,mid
count,269748.000000,269748.000000
mean,0.130862,222.717258
std,0.056788,1.398947
min,0.010000,220.520000
25%,0.090000,221.265000
50%,0.130000,222.655000
75%,0.160000,223.875000
max,0.770000,226.025000


In [23]:
# Interactive graph of best bid, best ask and all trades over time
import plotly.graph_objects as go

# Turn "seconds after midnight" into a real clock time for the x-axis
lob["clock"] = pd.Timestamp("2012-06-21") + pd.to_timedelta(lob["time"], unit="s")

# Most events don't move the best bid or ask, so keep only the rows where one of them changed.
# The step line looks exactly the same, but the chart is much lighter and faster.
changed = (lob["ask_p1"].diff() != 0) | (lob["bid_p1"].diff() != 0)
changed.iloc[-1] = True   # keep the last row so the lines run to the close
quotes = lob[changed]

# Trades are event types 4 (visible execution) and 5 (hidden execution)
trades = lob[lob["type"].isin([4, 5])]
print("quote changes:", len(quotes), " trades:", len(trades))

fig = go.Figure()

# Quotes are step lines ("hv"): a quote stays the same until the next event changes it
fig.add_trace(go.Scattergl(x=quotes["clock"], y=quotes["ask_p1"] / 10000, name="Best ask",
                           mode="lines", line=dict(color="#e34948", width=1.5, shape="hv")))
fig.add_trace(go.Scattergl(x=quotes["clock"], y=quotes["bid_p1"] / 10000, name="Best bid",
                           mode="lines", line=dict(color="#008300", width=1.5, shape="hv")))

# Each trade is a dot at the price it executed at; hover shows its size
fig.add_trace(go.Scattergl(x=trades["clock"], y=trades["price"] / 10000, name="Trades",
                           mode="markers", marker=dict(color="#0b0b0b", size=4, opacity=0.5),
                           customdata=trades["size"], hovertemplate="$%{y:.2f}, %{customdata} shares"))

fig.update_layout(
    title="AMZN best bid, best ask and trades, 21 Jun 2012",
    xaxis_title="Time", yaxis_title="Price (USD)",
    template="plotly_white", height=550,
    hovermode="x unified",      # one tooltip showing every series at the time under the mouse
    yaxis_tickprefix="$", yaxis_tickformat=".2f",
)
fig.show()

quote changes: 27558  trades: 11419


In [25]:
# Histogram of trades in half-hour blocks across the day
import plotly.graph_objects as go

# Trades are event types 4 (visible execution) and 5 (hidden execution)
trades = lob[lob["type"].isin([4, 5])]

# Clock time of each trade, rounded down to the start of its half-hour block (09:30, 10:00, ...)
trade_clock = pd.Timestamp("2012-06-21") + pd.to_timedelta(trades["time"], unit="s")
block_start = trade_clock.dt.floor("30min")

# Count the trades in each block
trades_per_block = block_start.value_counts().sort_index()
labels = trades_per_block.index.strftime("%H:%M")
print(trades_per_block.to_string())

fig = go.Figure(go.Bar(x=labels, y=trades_per_block.values, marker_color="#2a78d6",
                       hovertemplate="%{x} block: %{y} trades<extra></extra>"))
fig.update_layout(
    title="AMZN trades per half-hour block, 21 Jun 2012",
    xaxis_title="Block start time", yaxis_title="Number of trades",
    template="plotly_white", height=450, bargap=0.1,
)
fig.show()

time
2012-06-21 09:30:00     952
2012-06-21 10:00:00     892
2012-06-21 10:30:00    1312
2012-06-21 11:00:00     774
2012-06-21 11:30:00     420
2012-06-21 12:00:00     607
2012-06-21 12:30:00     557
2012-06-21 13:00:00     499
2012-06-21 13:30:00     585
2012-06-21 14:00:00     694
2012-06-21 14:30:00     593
2012-06-21 15:00:00    1133
2012-06-21 15:30:00    2401


In [26]:
lob["spread"] = lob["ask_p1"] - lob["bid_p1"]
lob.nlargest(30, "spread")[["time", "spread"]]

,time,spread
0,34200.017460,7700
1012,34334.776090,7500
1006,34334.650887,7400
1008,34334.650991,7400
1049,34341.811440,7300
1050,34341.861822,7300
1051,34341.867509,7300
1052,34341.867705,7300
1053,34341.876516,7300
1054,34341.876718,7300


In [27]:
day = lob[(lob["time"] >= 35100) & (lob["time"] < 56700)]   # trimmed 09:45–15:45
mid = (day["ask_p1"] + day["bid_p1"]) / 2
t = pd.to_datetime(day["time"], unit="s")

for label, series in [("event", mid),
                      ("1s",  mid.groupby(t.dt.floor("1s")).last()),
                      ("5s",  mid.groupby(t.dt.floor("5s")).last())]:
    print(label, f"{(series.diff().dropna() == 0).mean():.1%} zero returns, n={len(series)}")

event 90.1% zero returns, n=243429
1s 64.5% zero returns, n=17504
5s 31.9% zero returns, n=4289


In [28]:
day = lob[(lob["time"] >= 35100) & (lob["time"] < 56700)]   # trimmed 09:45–15:45
mid = (day["ask_p1"] + day["bid_p1"]) / 2
t = pd.to_datetime(day["time"], unit="s")

print("event", f"{(mid.diff().dropna() == 0).mean():.1%} zero returns, n={len(mid)}")

for freq in ["1s", "5s"]:
    s = mid.groupby(t.dt.floor(freq)).last()                     # last mid in each interval that had events
    full = pd.date_range(pd.to_datetime(35100, unit="s"),
                         pd.to_datetime(56700, unit="s"),
                         freq=freq, inclusive="left")            # every interval, including empty ones
    s = s.reindex(full).ffill()                                  # empty interval = book unchanged
    print(freq, f"{(s.diff().dropna() == 0).mean():.1%} zero returns, n={len(s)}")

event 90.1% zero returns, n=243429
1s 71.2% zero returns, n=21600
5s 32.4% zero returns, n=4320
